# Ensemble learning on Sri Lankan flood risk data

Compare bagging (Random Forest), boosting (XGBoost, LightGBM, CatBoost), and a hybrid stacking model (RF + XGBoost + LightGBM + ANN). The included data is synthetic; results are for coursework demonstration only.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'backend').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data' / 'sri_lanka_flood_risk_dataset_25000.csv'
df = pd.read_csv(DATA)
print('Shape:', df.shape)
display(df.head())

In [ ]:
# Review missing values, target distribution, and synthetic-data flag.
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().sort_values(ascending=False).head(12).to_frame('missing'))
display(df['flood_risk_score'].describe())
if 'is_synthetic' in df:
    display(df['is_synthetic'].value_counts(dropna=False))
df['flood_risk_score'].hist(bins=30)
plt.title('Flood risk score distribution')
plt.xlabel('Flood risk score')
plt.show()

## Leakage review
The flood occurrence, inundation area, and generated post-outcome judgement fields are excluded in `backend/preprocessing.py`. Check every predictor against the intended prediction time before relying on results.

In [ ]:
from backend.train_models import train
comparison = train(DATA, target='flood_risk_score', test_size=0.2, seed=42)
display(comparison)

In [ ]:
# Compare metrics; R2 higher is better, MSE/RMSE lower are better.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric in zip(axes, ['R2', 'MSE', 'RMSE']):
    comparison.sort_values(metric, ascending=(metric != 'R2')).plot.barh(
        x='Model', y=metric, ax=ax, legend=False, color='#0f766e')
    ax.set_title(metric)
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Interpretation prompts
1. Does the best model by R² also have the lowest MSE and RMSE?
2. Does stacking improve on the best individual model enough to justify its extra cost?
3. Which features have plausible predictive signal, and could any be leakage or proxies?
4. What changes would be needed for a location-grouped or time-based validation split?

Open the dashboard with `streamlit run dashboard/dashboard.py`. Training writes the same comparison and model bundle used by the UI.